In [1]:
# ===== L-1: Observability mask + Label_Flooded_Event_v02 =====
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(r"D:\Siam\Thesis_GeoAI")
P_MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
P_LABEL_V01 = ROOT / "03_features" / "Label_Flooded_Event_v01.csv"
P_LABEL_V02 = ROOT / "03_features" / "Label_Flooded_Event_v02.csv"
P_MASK = ROOT / "03_features" / "Observability_Mask_v01.csv"
TREE_MAX = 0.3    # observable if Tree_Fraction < TREE_MAX
MANG_MAX = 0.1    # observable if Mangrove_Fraction_GMW <= MANG_MAX
OVERWRITE = False
N_MODEL = 160874
N_LABEL_ROWS = 457251

for p in (P_LABEL_V02, P_MASK):
    assert OVERWRITE or not p.exists(), f"{p.name} exists; set OVERWRITE=True only if intended"

m = pd.read_csv(P_MASTER, usecols=["Grid_ID", "Is_Water", "Tree_Fraction",
                                    "Mangrove_Fraction_GMW", "Population_Count", "District"])
model = m.loc[m["Is_Water"] == 0].copy()
assert len(model) == N_MODEL and not model["Grid_ID"].duplicated().any()

lab = pd.read_csv(P_LABEL_V01)
assert len(lab) == N_LABEL_ROWS and not lab.duplicated(["Grid_ID", "Event_ID"]).any()

# --- Observability mask (all 160,874 model cells) ---
is_tree = model["Tree_Fraction"] >= TREE_MAX
is_mang = model["Mangrove_Fraction_GMW"] > MANG_MAX
model["Is_Observable"] = (~is_tree & ~is_mang).astype(int)
model["Unobservable_Reason"] = np.select(
    [is_tree & is_mang, is_tree, is_mang],
    ["Tree_And_Mangrove", "Tree_Canopy", "Mangrove"], default="Observable")
model["Has_Label_Row"] = model["Grid_ID"].isin(lab["Grid_ID"]).astype(int)
mask = model[["Grid_ID", "District", "Is_Observable", "Unobservable_Reason", "Has_Label_Row"]]

# --- Label v02 ---
n_not_model = (~lab["Grid_ID"].isin(model["Grid_ID"])).sum()
print("Label cells not in model cells:", n_not_model)
assert n_not_model == 0, "Some label cells are water cells; stop and paste this"
lab2 = lab.merge(mask[["Grid_ID", "Is_Observable", "Unobservable_Reason"]],
                 on="Grid_ID", how="left", validate="m:1")
lab2["Flooded_Event_Raw_v01"] = lab2["Flooded_Event"]
lab2["Flooded_Event"] = lab2["Flooded_Event"].where(lab2["Is_Observable"] == 1)  # NaN = unobservable
assert len(lab2) == N_LABEL_ROWS

# --- Checks ---
ev = lab2.groupby("Event_ID").agg(
    Rows=("Grid_ID", "size"),
    Rows_Observable=("Is_Observable", "sum"),
    Pos_Raw=("Flooded_Event_Raw_v01", "sum"),
    Pos_Kept=("Flooded_Event", "sum"),
    Rate_Observable_Pct=("Flooded_Event", "mean"))
ev["Pct_Pos_Kept"] = 100 * ev["Pos_Kept"] / ev["Pos_Raw"]
ev["Rate_Observable_Pct"] *= 100
print("\n=== By event ===\n", ev.round(3).to_string())

pop = model["Population_Count"]
print(f"\nObservable cells: {model['Is_Observable'].sum():,} of {N_MODEL:,} "
      f"({100 * model['Is_Observable'].mean():.2f}%)")
print(f"Observable AND labeled cells (training domain): "
      f"{((model['Is_Observable'] == 1) & (model['Has_Label_Row'] == 1)).sum():,}")
print(f"Population in observable cells: {100 * pop[model['Is_Observable'] == 1].sum() / pop.sum():.2f}%")
print("\nReason counts:\n", model["Unobservable_Reason"].value_counts().to_string())

d = lab2.groupby("District").agg(Obs_Share_Pct=("Is_Observable", "mean"),
                                 Rate_Observable_Pct=("Flooded_Event", "mean"),
                                 Positives_Kept=("Flooded_Event", "sum"))
d[["Obs_Share_Pct", "Rate_Observable_Pct"]] *= 100
print("\n=== By district ===\n", d.round(3).to_string())

lab2.to_csv(P_LABEL_V02, index=False)
mask.to_csv(P_MASK, index=False)
print("\nSaved:", P_LABEL_V02, "\nSaved:", P_MASK)
print("Label v02 columns:", list(lab2.columns))

Label cells not in model cells: 0

=== By event ===
             Rows  Rows_Observable  Pos_Raw  Pos_Kept  Rate_Observable_Pct  Pct_Pos_Kept
Event_ID                                                                               
Amphan    152417            49573     4022    3920.0                7.908        97.464
Remal     152417            49573    14138   13636.0               27.507        96.449
Yaas      152417            49573     2358    2326.0                4.692        98.643

Observable cells: 56,174 of 160,874 (34.92%)
Observable AND labeled cells (training domain): 49,573
Population in observable cells: 19.53%

Reason counts:
 Unobservable_Reason
Tree_Canopy          74500
Observable           56174
Mangrove             29017
Tree_And_Mangrove     1183

=== By district ===
             Obs_Share_Pct  Rate_Observable_Pct  Positives_Kept
District                                                      
Bagerhat           20.887               14.250          4440.0
Barguna    